In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import sys
import os
from pathlib import Path
from datetime import datetime

project_root = Path.cwd().parent.parent
project_root_str = str(project_root)

if project_root_str not in sys.path:
    sys.path.insert(0, project_root_str)

os.chdir(project_root_str)

from src.features.features_v1 import *
from src.features.features_v2 import *
from src.pipeline.calculate_evs import *
from src.utils.helper_functions import *
from src.utils.team_info import teamStarPlayer, projectedStartingFive, mainStartingFive

### Update projected starting lineups

In [2]:
from src.utils.scrap_starters import NBADailyLineups

scraper = NBADailyLineups("https://www.rotowire.com/basketball/nba-lineups.php")
scraper.getDict()  # Scrape the lineups
scraper.updateTeamInfo()  # Update teamInfo.py

Successfully updated /Users/alexgonzalez/Documents/NBA-Prop-Predictor/src/utils/team_info.py
Updated 6 teams with confirmed lineups


### Load Model

### Load Player Data and Bookmaker Data

In [2]:
pd.set_option('display.max_columns', None)
today = datetime.today().strftime('%Y%m%d')  
current_date = datetime.now().strftime('%Y-%m-%d')

def get_latest_file(pattern):
    files = list(Path('data/raw/player_lines').glob(pattern))
    if not files:
        return None
    return max(files, key=lambda p: p.stat().st_mtime)

us_file = get_latest_file(f'NBA_US_{today}*.csv')
dfs_file = get_latest_file(f'NBA_DFS_{today}*.csv')

if us_file is None:
    raise FileNotFoundError(f"No NBA_US file found for {today}")
if dfs_file is None:
    raise FileNotFoundError(f"No NBA_DFS file found for {today}")

s26 = pd.read_csv('data/processed/training/PTS_TRAIN_26.csv').sort_values(by='GAME_DATE')
usData = pd.read_csv(us_file)
dfsData = pd.read_csv(dfs_file)

print(f"Loaded: {us_file.name}")
print(f"Loaded: {dfs_file.name}")
dfsData.head()

Loaded: NBA_US_20251208_142444.csv
Loaded: NBA_DFS_20251208_142406.csv


,BOOKMAKER,CATEGORY,NAME,OVER/UNDER,LINE,ODDS,COMMENCE_TIME,LAST_UPDATE,DATA_PULLED_AT
0,Underdog,player_points,Pascal Siakam,Over,24.5,-137,2025-12-09,2025-12-08T22:23:21Z,2025-12-08 14:24:06
1,Underdog,player_points,Pascal Siakam,Under,24.5,-137,2025-12-09,2025-12-08T22:23:21Z,2025-12-08 14:24:06
2,Underdog,player_points,Zach LaVine,Over,21.5,-137,2025-12-09,2025-12-08T22:23:21Z,2025-12-08 14:24:06
3,Underdog,player_points,Zach LaVine,Under,21.5,-137,2025-12-09,2025-12-08T22:23:21Z,2025-12-08 14:24:06
4,Underdog,player_points,Andrew Nembhard,Over,16.5,-137,2025-12-09,2025-12-08T22:23:21Z,2025-12-08 14:24:06


In [3]:
from src.features.feature_engine import FeatureEngine

engine = FeatureEngine({
    "min_model": "src/models/saved/min_model.pkl",
    "usg_model": "src/models/saved/usg_model.pkl",
    "fga_model": "src/models/saved/fga_model.pkl",
    "ngboost_model_wrapper": "src/models/saved/pts_model_wrapper.pkl"
})

/Users/alexgonzalez/Documents/NBA-Prop-Predictor/nba_model/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Top EVs for 2 leg bets

### Underdog picks

In [4]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points')]

underdogPairs = calculate2LegBets(
    data=s26,
    bookmakers=dfsPTS,
    engine=engine,
    current_date=current_date,
    top_n=10,                    # Get top 10 bets
    max_player_appearances=1,    # Each player appears max once
    projectedStartingFive=projectedStartingFive,
    mainStartingFive=mainStartingFive,
    teamStarPlayer=teamStarPlayer,
    league_df=league_df,
    findOpp=findOpp
)

underdogPairs = underdogPairs[['NAME 1', 'NAME 2', 'LINE 1', 'LINE 2','ODDS 1', 'ODDS 2','PREDICTION 1', 'PREDICTION 2', 'MODEL_PROB 1', 'MODEL_PROB 2', 'SIDE 1', 'SIDE 2', 'PARLAY_PROB', 'PARLAY_ODDS', 'EV_PERCENT', 'KELLY_QUARTER']]
underdogPairs.to_csv('data/props/ev_analysis/underdogPairs.csv', index=False)
underdogPairs

Computing predictions for 25 players...
Found 25 valid players
Generated 206 valid 2-leg combinations


,NAME 1,NAME 2,LINE 1,LINE 2,ODDS 1,ODDS 2,PREDICTION 1,PREDICTION 2,MODEL_PROB 1,MODEL_PROB 2,SIDE 1,SIDE 2,PARLAY_PROB,PARLAY_ODDS,EV_PERCENT,KELLY_QUARTER
5,Pascal Siakam,Grayson Allen,24.5,18.5,-104,-104,27.85,14.24,0.702,0.665,over,under,0.467,285,79.65,0.0699
194,Ryan Dunn,Saddiq Bey,6.5,16.5,-110,-107,7.94,12.29,0.672,0.675,over,under,0.454,269,67.39,0.0626
105,Malik Monk,Jordan Hawkins,13.5,6.5,102,-108,15.00,8.14,0.621,0.660,over,over,0.410,289,59.57,0.0515
157,Dillon Brooks,Derik Queen,22.5,13.5,-108,-110,24.68,10.32,0.648,0.623,over,under,0.404,268,48.78,0.0455
135,Julius Randle,Jeremy Sochan,22.5,6.5,-109,-110,19.15,7.42,0.618,0.608,under,over,0.376,266,37.44,0.0352
49,Andrew Nembhard,Trey Murphy III,16.5,21.5,-103,-105,16.78,19.27,0.554,0.540,over,under,0.299,285,15.11,0.0133
27,Zach LaVine,Oso Ighodaro,21.5,4.5,-120,100,18.57,2.82,0.580,0.525,under,under,0.304,267,11.74,0.0110
76,Bennedict Mathurin,Jaden McDaniels,21.5,15.5,-102,-107,19.51,13.99,0.527,0.501,under,under,0.264,283,1.16,0.0010
56,Russell Westbrook,Donte DiVincenzo,12.5,13.5,112,100,11.57,12.29,0.458,0.482,under,under,0.221,324,-6.45,0.0000
132,Anthony Edwards,Jeremiah Fears,28.5,16.5,-105,-107,27.34,15.50,0.483,0.467,under,under,0.226,278,-14.62,0.0000


### Prizepicks picks

In [6]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points')]

prizepicksPairs = calculate2LegBets(
    data=s26,
    bookmakers=dfsPTS,
    engine=engine,
    current_date=current_date,
    top_n=10,                    # Get top 10 bets
    max_player_appearances=1,    # Each player appears max once
    projectedStartingFive=projectedStartingFive,
    mainStartingFive=mainStartingFive,
    teamStarPlayer=teamStarPlayer,
    league_df=league_df,
    findOpp=findOpp
)

prizepicksPairs = prizepicksPairs[['NAME 1', 'NAME 2', 'LINE 1', 'LINE 2','ODDS 1', 'ODDS 2','PREDICTION 1', 'PREDICTION 2', 'MODEL_PROB 1', 'MODEL_PROB 2', 'SIDE 1', 'SIDE 2', 'PARLAY_PROB', 'PARLAY_ODDS', 'EV_PERCENT', 'KELLY_QUARTER']]
prizepicksPairs.to_csv('data/props/ev_analysis/prizepicksPairs.csv', index=False)
prizepicksPairs

Computing predictions for 36 players...
[MIN] No data found for Herb Jones
Found 35 valid players
Generated 398 valid 2-leg combinations


,NAME 1,NAME 2,LINE 1,LINE 2,ODDS 1,ODDS 2,PREDICTION 1,PREDICTION 2,MODEL_PROB 1,MODEL_PROB 2,SIDE 1,SIDE 2,PARLAY_PROB,PARLAY_ODDS,EV_PERCENT,KELLY_QUARTER
209,Maxime Raynaud,Dylan Harper,12.0,12.5,-137,102,4.63,16.96,0.882,0.807,under,over,0.713,249,148.68,0.1493
13,Pascal Siakam,De'Aaron Fox,24.5,24.5,-104,108,27.85,29.46,0.702,0.778,over,over,0.546,308,122.84,0.0997
277,Collin Gillespie,Stephon Castle,17.0,14.5,-137,120,9.73,16.48,0.757,0.656,under,over,0.497,281,89.36,0.0795
261,Grayson Allen,Saddiq Bey,18.5,16.5,-104,-107,14.24,12.29,0.665,0.675,under,under,0.449,279,70.05,0.0628
374,Ryan Dunn,Devin Vassell,6.5,15.5,-110,-115,7.94,18.59,0.672,0.697,over,over,0.468,257,67.20,0.0654
187,Malik Monk,Jordan Hawkins,13.5,6.5,102,-108,15.00,8.14,0.621,0.660,over,over,0.410,289,59.57,0.0515
239,Dillon Brooks,Keldon Johnson,22.5,12.5,-108,-105,24.68,13.70,0.648,0.620,over,over,0.402,276,51.26,0.0464
250,Julius Randle,Derik Queen,22.5,13.5,-109,-110,19.15,10.32,0.618,0.623,under,under,0.385,266,41.02,0.0385
355,Rudy Gobert,Jeremy Sochan,10.5,6.5,-103,-110,7.92,7.42,0.594,0.608,under,over,0.361,276,35.73,0.0324
122,Andrew Nembhard,Trey Murphy III,16.5,21.5,-103,-105,16.78,19.27,0.554,0.540,over,under,0.299,285,15.11,0.0133


## 3 leg parlay

### Underdog picks

In [8]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points') ]

underdogTrios = calculate3LegBets(
    data=s26,
    bookmakers=dfsPTS,
    engine=engine,
    current_date=current_date,
    top_n=10,                    # Get top 10 bets
    max_player_appearances=1,    # Each player appears max once
    projectedStartingFive=projectedStartingFive,
    mainStartingFive=mainStartingFive,
    teamStarPlayer=teamStarPlayer,
    league_df=league_df,
    findOpp=findOpp
)

underdogTrios = underdogTrios[['NAME 1', 'NAME 2', 'NAME 3', 'LINE 1', 'LINE 2', 'LINE 3', 'ODDS 1', 'ODDS 2', 'ODDS 3', 'PREDICTION 1', 'PREDICTION 2', 'PREDICTION 3', 'MODEL_PROB 1', 'MODEL_PROB 2', 'MODEL_PROB 3', 'SIDE 1', 'SIDE 2', 'SIDE 3', 'PARLAY_PROB', 'PARLAY_ODDS', 'EV_PERCENT', 'KELLY_QUARTER']]
underdogTrios.to_csv('data/props/ev_analysis/underdogTrios.csv', index=False)
underdogTrios.head()

Computing predictions for 25 players...
Found 25 valid players
Generated 560 valid 3-leg combinations


,NAME 1,NAME 2,NAME 3,LINE 1,LINE 2,LINE 3,ODDS 1,ODDS 2,ODDS 3,PREDICTION 1,PREDICTION 2,PREDICTION 3,MODEL_PROB 1,MODEL_PROB 2,MODEL_PROB 3,SIDE 1,SIDE 2,SIDE 3,PARLAY_PROB,PARLAY_ODDS,EV_PERCENT,KELLY_QUARTER
44,Pascal Siakam,Grayson Allen,Saddiq Bey,24.5,18.5,16.5,-104,-104,-107,27.85,14.24,12.29,0.702,0.665,0.675,over,under,under,0.315,644,134.22,0.0521
469,Malik Monk,Ryan Dunn,Jordan Hawkins,13.5,6.5,6.5,102,-110,-108,15.00,7.94,8.14,0.621,0.672,0.660,over,over,over,0.276,643,104.93,0.0408
191,Andrew Nembhard,Dillon Brooks,Derik Queen,16.5,22.5,13.5,-103,-108,-110,16.78,24.68,10.32,0.554,0.648,0.623,over,over,under,0.224,625,62.32,0.0249
89,Zach LaVine,Julius Randle,Jeremy Sochan,21.5,22.5,6.5,-120,-109,-110,18.57,19.15,7.42,0.580,0.618,0.608,under,under,over,0.218,571,46.18,0.0202
395,Bennedict Mathurin,Oso Ighodaro,Trey Murphy III,21.5,4.5,21.5,-102,100,-105,19.51,2.82,19.27,0.527,0.525,0.540,under,under,under,0.149,673,15.46,0.0057


### Prizepicks picks

In [9]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points')]

triosPrizepicks = calculate3LegBets(
    data=s26,
    bookmakers=dfsPTS,
    engine=engine,
    current_date=current_date,
    top_n=10,                    # Get top 10 bets
    max_player_appearances=1,    # Each player appears max once
    projectedStartingFive=projectedStartingFive,
    mainStartingFive=mainStartingFive,
    teamStarPlayer=teamStarPlayer,
    league_df=league_df,
    findOpp=findOpp
)

triosPrizepicks = triosPrizepicks[['NAME 1', 'NAME 2', 'NAME 3', 'LINE 1', 'LINE 2', 'LINE 3', 'ODDS 1', 'ODDS 2', 'ODDS 3', 'PREDICTION 1', 'PREDICTION 2', 'PREDICTION 3', 'MODEL_PROB 1', 'MODEL_PROB 2', 'MODEL_PROB 3', 'SIDE 1', 'SIDE 2', 'SIDE 3', 'PARLAY_PROB', 'PARLAY_ODDS', 'EV_PERCENT', 'KELLY_QUARTER']]
triosPrizepicks.to_csv('data/props/ev_analysis/prizepicksTrios.csv', index=False)
triosPrizepicks.head()

Computing predictions for 36 players...
[MIN] No data found for Herb Jones
Found 35 valid players
Generated 1456 valid 3-leg combinations


,NAME 1,NAME 2,NAME 3,LINE 1,LINE 2,LINE 3,ODDS 1,ODDS 2,ODDS 3,PREDICTION 1,PREDICTION 2,PREDICTION 3,MODEL_PROB 1,MODEL_PROB 2,MODEL_PROB 3,SIDE 1,SIDE 2,SIDE 3,PARLAY_PROB,PARLAY_ODDS,EV_PERCENT,KELLY_QUARTER
1337,Maxime Raynaud,Collin Gillespie,Dylan Harper,12.0,17.0,12.5,-137,-137,102,4.63,9.73,16.96,0.882,0.757,0.807,under,under,over,0.540,505,226.44,0.1121
42,Pascal Siakam,Grayson Allen,De'Aaron Fox,24.5,18.5,24.5,-104,-104,108,27.85,14.24,29.46,0.702,0.665,0.778,over,under,over,0.363,700,190.59,0.0681
1251,Malik Monk,Ryan Dunn,Stephon Castle,13.5,6.5,14.5,102,-110,120,15.00,7.94,16.48,0.621,0.672,0.656,over,over,over,0.274,748,132.43,0.0443
745,Andrew Nembhard,Dillon Brooks,Saddiq Bey,16.5,22.5,16.5,-103,-108,-107,16.78,24.68,12.29,0.554,0.648,0.675,over,over,under,0.242,634,77.82,0.0307
396,Zach LaVine,Julius Randle,Devin Vassell,21.5,22.5,15.5,-120,-109,-115,18.57,19.15,18.59,0.580,0.618,0.697,under,under,over,0.250,557,64.06,0.0288
